# S6E10 — 07 TabPFN members from the public dataset

**What this is:** TabPFN-3.5 is a pre-trained transformer that classifies rows by reading labelled rows as context. A full five-fold run with the whole 560k-row context needs a machine far bigger than ours, so we use the predictions that goodpjw2008 published as the Kaggle dataset **`goodpjw2008/s6e10-tabpfn-member`** (credit to the author and to hemingweb / Prior Labs for the idea and the model). The predictions are out-of-fold on the same split as ours (`StratifiedKFold(5, shuffle=True, random_state=42)` on the competition train order).

**Before any use we verify the alignment:** the OOF AUC and the per-fold AUCs, recomputed on *our* labels and *our* folds, must match the numbers in the dataset's `meta.json`. If they do not, the rows are not aligned and the members must not go into the stack.

**Output:** `preds/oof_tabpfn_avg.npy` / `preds/test_tabpfn_avg.npy` (probabilities) — the logit average of the five route-aware members (they are very similar to each other, so one averaged member is better for the stack than five). `05_stack` picks it up automatically.

**How to get the data:** open https://www.kaggle.com/datasets/goodpjw2008/s6e10-tabpfn-member, press *Download*, unzip into `data/external/s6e10-tabpfn-member/` next to this notebook (the folder must contain `meta.json` and `oof_*.csv` / `test_*.csv`). On Kaggle: attach the dataset via *Add Input*.

In [1]:
import json, glob
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

N_FOLDS, RANDOM_STATE = 5, 42
RAW = next(p for p in [Path('data/raw'), Path('../data/raw'), Path('/kaggle/input/playground-series-s6e10')] if (p / 'train.csv').exists())
PROC = next(p for p in [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')] if (p / 'y_train.parquet').exists())
PREDS_DIR = next((p for p in [Path('preds'), Path('../preds'), Path('/kaggle/working/preds')] if p.exists()), Path('preds')); PREDS_DIR.mkdir(exist_ok=True)

meta_hits = sorted(glob.glob('data/external/**/meta.json', recursive=True) + glob.glob('../data/external/**/meta.json', recursive=True)
                   + glob.glob('/kaggle/input/**/s6e10-tabpfn-member/**/meta.json', recursive=True) + glob.glob('/kaggle/input/**/meta.json', recursive=True))
assert meta_hits, 'meta.json not found: download the dataset into data/external/s6e10-tabpfn-member/ (see the text above)'
DS = Path(meta_hits[0]).parent
meta = json.load(open(DS / 'meta.json'))
print('dataset folder:', DS)
print('members in meta.json:', list(meta['members']))

train_ids = pd.read_csv(RAW / 'train.csv', usecols=['id'])['id'].values
test_ids = pd.read_csv(RAW / 'test.csv', usecols=['id'])['id'].values
y = pd.read_parquet(PROC / 'y_train.parquet')['y'].astype(int).values
assert len(train_ids) == len(y) == 699635 and len(test_ids) == 299844
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(np.zeros(len(y)), y))

dataset folder: data/external/s6e10-tabpfn-member
members in meta.json: ['tabpfn_full_raw', 'tabpfn_full_raw_s1', 'tabpfn_te', 'tabpfn_te4', 'tabpfn_catfd', 'tabpfn_catfd_s1', 'tabpfn_catfd10', 'tabicl_full_raw']


## 1. Load and verify every member

In [2]:
OOF, TEST, rows = {}, {}, []
for name in meta['members']:
    fo, ft = DS / f'oof_{name}.csv', DS / f'test_{name}.csv'
    if not (fo.exists() and ft.exists()): print('skipped (files missing):', name); continue
    o = pd.read_csv(fo).set_index('id')['pred']; t = pd.read_csv(ft).set_index('id')['pred']
    o, t = o.reindex(train_ids).values, t.reindex(test_ids).values
    assert not np.isnan(o).any() and not np.isnan(t).any(), f'{name}: ids do not cover our train/test'
    ours_fold = [roc_auc_score(y[va], o[va]) for _, va in FOLDS]
    theirs_fold = meta['members'][name]['fold_auc']
    OOF[name], TEST[name] = o, t
    rows.append({'member': name, 'OOF AUC (ours)': round(roc_auc_score(y, o), 5), 'OOF AUC (meta)': round(meta['members'][name]['oof_auc'], 5),
                 'max |fold diff|': round(max(abs(a - b) for a, b in zip(ours_fold, theirs_fold)), 6)})
tab = pd.DataFrame(rows); print(tab.to_string(index=False))
bad = tab[tab['max |fold diff|'] > 2e-4]
print('\nALIGNMENT OK: every member matches its meta.json on our labels and folds' if bad.empty else f'\nWARNING: not aligned: {bad.member.tolist()} -> do not use these members')

            member  OOF AUC (ours)  OOF AUC (meta)  max |fold diff|
   tabpfn_full_raw         0.96073         0.96073              0.0
tabpfn_full_raw_s1         0.96063         0.96063              0.0
         tabpfn_te         0.96110         0.96110              0.0
        tabpfn_te4         0.96114         0.96114              0.0
      tabpfn_catfd         0.96092         0.96092              0.0
   tabpfn_catfd_s1         0.96089         0.96089              0.0
    tabpfn_catfd10         0.96107         0.96107              0.0
   tabicl_full_raw         0.95759         0.95759              0.0

ALIGNMENT OK: every member matches its meta.json on our labels and folds


## 2. Average the route-aware members and save
The five route-aware members (`tabpfn_te`, `tabpfn_te4`, `tabpfn_catfd`, `tabpfn_catfd_s1`, `tabpfn_catfd10`) are averaged on the logit scale. The raw-column members are saved separately as `tabpfn_raw_avg` and TabICL v2 as `tabicl` (both weaker, a different view; `05_stack` uses them as separate members).

In [3]:
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
sg = lambda z: 1 / (1 + np.exp(-z))
ROUTE = [m for m in ['tabpfn_te', 'tabpfn_te4', 'tabpfn_catfd', 'tabpfn_catfd_s1', 'tabpfn_catfd10'] if m in OOF and m not in (bad.member.tolist() if not bad.empty else [])]
RAWM = [m for m in ['tabpfn_full_raw', 'tabpfn_full_raw_s1'] if m in OOF and m not in (bad.member.tolist() if not bad.empty else [])]
def save_avg(members, tag):
    o = np.mean([lg(OOF[m]) for m in members], axis=0); t = np.mean([lg(TEST[m]) for m in members], axis=0)
    np.save(PREDS_DIR / f'oof_{tag}.npy', sg(o)); np.save(PREDS_DIR / f'test_{tag}.npy', sg(t))
    print(f'{tag}: {len(members)} members, OOF AUC {roc_auc_score(y, o):.5f}, per fold', [round(roc_auc_score(y[va], o[va]), 5) for _, va in FOLDS], '-> saved')
    return o
o_route = save_avg(ROUTE, 'tabpfn_avg') if ROUTE else None
o_raw = save_avg(RAWM, 'tabpfn_raw_avg') if RAWM else None
o_icl = save_avg(['tabicl_full_raw'], 'tabicl') if 'tabicl_full_raw' in OOF and 'tabicl_full_raw' not in (bad.member.tolist() if not bad.empty else []) else None

# correlation with our own models (if their OOF files are in preds/)
from scipy.stats import spearmanr
for nm in ['realmlp', 'xgb', 'cat_v1', 'mlp']:
    f = PREDS_DIR / f'oof_{nm}.npy'
    if f.exists() and o_route is not None: print(f'Spearman tabpfn_avg vs {nm}: {spearmanr(o_route, np.load(f))[0]:.4f}')

tabpfn_avg: 5 members, OOF AUC 0.96158, per fold [0.96167, 0.96081, 0.96212, 0.96183, 0.96149] -> saved
tabpfn_raw_avg: 2 members, OOF AUC 0.96089, per fold [0.96082, 0.9601, 0.96148, 0.96122, 0.96086] -> saved
tabicl: 1 members, OOF AUC 0.95759, per fold [0.9574, 0.95693, 0.95813, 0.95785, 0.95766] -> saved
Spearman tabpfn_avg vs realmlp: 0.9720
Spearman tabpfn_avg vs xgb: 0.9690
Spearman tabpfn_avg vs cat_v1: 0.9705
Spearman tabpfn_avg vs mlp: 0.9678


## 3. Next
Rerun `05_stack.ipynb`: it already looks for `oof_tabpfn_avg.npy` and uses it as the member `tabpfn`. Decide by the nested-CV table (gain over the stack without it, per fold), not by the leaderboard.